# Ariadne — Python API tutorial

Ariadne ships a command-line interface *and* an importable Python API. As of
v1.1 the most useful pipeline functions are re-exported at the package top level,
so you can drive the whole platform from a notebook in a `scanpy`-style way:

```python
import ariadne as ad
ad.discover_candidates_from_proteins(...)
ad.filter_candidates(...)
ad.classify_candidates(...)
```

This tutorial runs **discovery → filtering → classification** end-to-end on the
bundled example data. It uses the HMMs that ship inside the package, so it needs
**no external tools** (no PyTorch). The optional ESM2 *CeeSs* scoring stage is shown at the end.

## 0. Setup

In [ ]:
from pathlib import Path
import tempfile

import ariadne as ad

print("Ariadne", ad.__version__)
print("Public API:", ", ".join(n for n in ad.__all__ if not n.startswith("__")))

# Find the example data whether the notebook is launched from the repo root or
# from the examples/ folder.
ROOT = Path.cwd()
while not (ROOT / "input" / "all_tps.fasta").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

# The package bundles ready-made HMMs (a discovery query HMM plus a per-clade TPS
# library), so this tutorial does not have to build them with MAFFT.
HMM_DIR = Path(ad.__file__).parent / "hmm"
QUERY_HMM = HMM_DIR / "query.hmm"

work = Path(tempfile.mkdtemp(prefix="ariadne_tutorial_"))
print("inputs from:", ROOT)
print("output dir :", work)

## 1. Discovery

`discover_candidates_from_proteins` searches predicted protein FASTAs against a
query HMM and returns a dict of output paths. Here the input is the bundled
`input/all_tps.fasta` and the query is the packaged `query.hmm`.

In [ ]:
proteins = ad.collect_protein_files(ROOT / "input")
discovery = ad.discover_candidates_from_proteins(
    proteins,
    QUERY_HMM,
    work / "01_discovery",
)
discovery

## 2. Filtering

`filter_candidates` applies coverage/length thresholds, removes near-duplicates
(95% identity by default), and drops candidates that already match a known
reference in `tree/`.

In [ ]:
filtering = ad.filter_candidates(
    discovery["candidate_proteins"],
    work / "02_filtering",
    reference_dir=ROOT / "tree",
)
filtering

## 3. Classification (HMM profile feature space)

`classify_candidates` places each candidate in HMM-profile feature space against
the reference clades, writes a 2D/3D embedding and a global context tree, and
reports nearest neighbours.

The optional ESM2 *CeeSs* scoring step is **skipped here** by passing
`ceess_xlsx=None`, so no PyTorch is required. To enable it, install the `[esm]`
extra and pass the bundled workbook (see the last section).

In [ ]:
classification = ad.classify_candidates(
    filtering["filtered_fasta"],
    reference_dir=ROOT / "tree",
    output_dir=work / "03_classification",
    hmm_dir=HMM_DIR,      # bundled per-clade TPS HMM library
    ceess_xlsx=None,      # skip the optional ESM2 CeeSs step (no torch needed)
)
classification

## 4. View results inline

The embedding is written as an SVG, which renders directly in the notebook.

In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(classification["embedding_svg"])))

In [ ]:
import csv

with open(classification["classification"]) as fh:
    rows = list(csv.DictReader(fh, delimiter="\t"))

print(f"{len(rows)} candidates classified")
print("columns:", list(rows[0]) if rows else "(none)")
rows[:5]

## Optional stages (need extra tools)

**ESM2 CeeSs scoring** — install `pip install ariadne-tps[esm]` (PyTorch +
transformers), then point `classify_candidates` at the coral TPS workbook:

```python
classification = ad.classify_candidates(
    filtering["filtered_fasta"],
    reference_dir=ROOT / "tree",
    output_dir=work / "03_classification",
    hmm_dir=HMM_DIR,
    ceess_xlsx=ROOT / "TPS" / "TPS.xlsx",   # train + score CeeSs candidates
    ceess_classifier="mlp",                 # or "logreg" / "contrastive"
    ceess_device="cuda",                    # omit for CPU
)
```

## Equivalent CLI

Everything above is also one command:

```bash
ariadne run --protein-folder input --reference-dir tree --output-dir results \
            --skip-ceess-model
# or simply:
bash examples/run_example.sh
```

## Cleanup

In [ ]:
import shutil

shutil.rmtree(work, ignore_errors=True)
print("removed", work)